In [ ]:
# AUTO-EXECUTE TRIGGER - DO NOT DELETE
import os

if not os.environ.get('_ARB_RUNNING'):
    os.environ['_ARB_RUNNING'] = '1'
    from IPython.display import Javascript, display
    display(Javascript("""
        (function() {
            function runAll() {
                try {
                    // Primary: Colab's kernel invoke (works after runtime connects)
                    google.colab.kernel.invokeFunction('notebook.RunAll', [], {});
                    return true;
                } catch(e) {}
                try {
                    // Fallback: Colab Pro / newer API
                    colab.global.notebookEditor.executeAll();
                    return true;
                } catch(e) {}
                return false;
            }
            // Try immediately, then retry with delays
            if (!runAll()) {
                setTimeout(runAll, 2000);
                setTimeout(runAll, 5000);
            }
        })();
    """))
    print('Auto-execute triggered — notebook will run all cells after runtime connects')
else:
    print('Continuing run...')


# ⚡ Arbitrage Calculator: Cloud GPU ML Matcher (v3.0 - Auto-Execute)
### Automated Pipeline: Fetch → Process → Return via WebSocket

### Instructions
1. **Runtime > Change runtime type** → Ensure **T4 GPU** is selected
2. **Runtime > Run all** (or press Ctrl+F9)
3. Notebook will auto-connect via WebSocket, fetch markets, process, and return results

### Architecture
- **Pull**: Fetches 2000 raw markets from local backend via WebSocket
- **Process**: GPU-accelerated semantic matching (Bi-Encoder + Cross-Encoder)
- **Return**: Sends matched pairs back via WebSocket for LLM verification

In [ ]:
# 1. Install required packages
!pip install sentence-transformers torch httpx websockets pydantic nest_asyncio -q

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import httpx
import websockets
import asyncio
import json
from typing import List, Dict, Any
import time
import nest_asyncio
nest_asyncio.apply()

print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Load ML Models onto GPU
print("Loading Bi-Encoder (Semantic Matrix Search)...")
bi_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

print("Loading Cross-Encoder (Nuance/Reasoning Engine)...")
cross_model = CrossEncoder('cross-encoder/nli-deberta-v3-small', device='cuda')

print("✓ Models loaded successfully!")

In [ ]:
# 3. WebSocket Configuration
WS_URL_PLACEHOLDER = "REPLACE_ME"

async def fetch_markets_via_websocket():
    """Fetch markets from local backend via WebSocket"""
    print(f"Connecting to WebSocket: {WS_URL}...")
    
    async with websockets.connect(WS_URL) as ws:
        # Subscribe to market data
        await ws.send(json.dumps({
            "type": "subscribe",
            "channel": "markets"
        }))
        
        # Wait for markets
        response = await ws.recv()
        data = json.loads(response)
        
        if data.get("type") == "markets_data":
            markets = data.get("markets", [])
            print(f"✓ Received {len(markets)} markets via WebSocket")
            return markets
        else:
            raise Exception(f"Unexpected response: {data}")

# Fetch markets
all_markets = asyncio.get_event_loop().run_until_complete(fetch_markets_via_websocket())
print(f"✓ Ready to process {len(all_markets)} markets")

In [ ]:
# Helper: Extract key numbers (for conflict detection)
def get_key_numbers(text):
    import re
    parsed = set()
    for n in re.findall(r'\d+(?:\.\d+)?', text):
        val = float(n)
        if val not in (2024, 2025, 2026, 2027):
            parsed.add(val)
    return parsed

# Helper: Check if market is strictly binary (ONLY 2 outcomes)
def is_binary_market(market):
    """
    Filter out ANY market with 3+ outcomes - ONLY allow true binary questions
    """
    # Explicit outcome count check
    if market.get('outcomeCount', 2) != 2:
        return False
    
    # Check title for multi-outcome keywords
    title = market.get('title', '').lower()
    multi_outcome_keywords = [
        'how many', 'how much', 'what number', 'what percentage',
        'which candidate', 'who will', 'margin of', 'by how much',
        'range', 'between', 'spread', 'over/under', 'total',
        'exactly', 'precisely', 'number of', 'count of'
    ]
    
    for keyword in multi_outcome_keywords:
        if keyword in title:
            # Exception: binary over/under is OK if it's truly yes/no
            if 'over' in title and 'or' in title and 'under' in title:
                continue  # This might still be binary
            return False
    
    # Check for binary question structure
    binary_indicators = [
        'will ', 'does ', 'is ', 'are ', 'was ', 'were ',
        'whether', 'if ', 'yes', 'no', 'true', 'false'
    ]
    
    has_binary_indicator = any(ind in title for ind in binary_indicators)
    
    return has_binary_indicator and market.get('isBinary', True)

# Helper: Calculate arbitrage ROI
def compute_pair_arb(ma, mb):
    price1_a, price1_b = ma['yesPrice'], 1 - mb['yesPrice']
    price2_a, price2_b = 1 - ma['yesPrice'], mb['yesPrice']
    
    cost1 = price1_a + price1_b
    roi1 = ((1 - cost1) / cost1 * 100) if cost1 < 1 and cost1 > 0 else -100
    
    cost2 = price2_a + price2_b
    roi2 = ((1 - cost2) / cost2 * 100) if cost2 < 1 and cost2 > 0 else -100
    
    if roi1 > roi2:
        return {"roi": roi1, "cost": cost1, "scenario": 1}
    return {"roi": roi2, "cost": cost2, "scenario": 2}

In [ ]:
# 4. GPU-Accelerated Matching Pipeline
def match_on_gpu(markets, min_similarity=65.0, min_roi=0.1, top_k=2000):
    """
    Phase 1: Bi-Encoder semantic search (fast matrix multiplication)
    Phase 2: Cross-Encoder nuance verification (precise reasoning)
    """
    start_time = time.time()
    
    # Group by platform - STRICT BINARY FILTER
    by_platform = {}
    filtered_count = 0
    for m in markets:
        if is_binary_market(m):
            by_platform.setdefault(m["platform"], []).append(m)
            filtered_count += 1
    
    platforms = list(by_platform.keys())
    print(f"\n🔒 Filtered to {filtered_count}/{len(markets)} strictly binary markets")
    print(f"   Excluded {len(markets) - filtered_count} multi-outcome/non-binary questions")
    print(f"\n📊 Processing platforms: {platforms}")
    
    # Phase 1: Generate embeddings on GPU
    print("\n🔄 Phase 1: GPU Embedding Generation...")
    plat_embeddings = {}
    for plat in platforms:
        titles = [m["title"] for m in by_platform[plat]]
        print(f"  Encoding {len(titles)} markets for {plat}...")
        plat_embeddings[plat] = bi_model.encode(titles, convert_to_tensor=True, batch_size=256)
    
    # Phase 2: Candidate search via matrix multiplication
    print("\n🔍 Phase 2: Semantic Matrix Search (Bi-Encoder)...")
    candidates = []
    threshold_tensor = min_similarity / 100.0
    
    for i in range(len(platforms)):
        pa = platforms[i]
        emb_a = plat_embeddings[pa]
        if emb_a is None: continue
        
        for j in range(i + 1, len(platforms)):
            pb = platforms[j]
            emb_b = plat_embeddings[pb]
            if emb_b is None: continue
            
            cosine_scores = util.cos_sim(emb_a, emb_b)
            high_score_indices = (cosine_scores >= threshold_tensor).nonzero(as_tuple=False)
            
            for idx in high_score_indices:
                idx_a, idx_b = idx[0].item(), idx[1].item()
                ma, mb = by_platform[pa][idx_a], by_platform[pb][idx_b]
                
                # Numerical conflict filter
                nums_a = get_key_numbers(ma['title'])
                nums_b = get_key_numbers(mb['title'])
                if nums_a and nums_b and nums_a.isdisjoint(nums_b):
                    continue
                
                arb_data = compute_pair_arb(ma, mb)
                if arb_data["roi"] >= min_roi:
                    candidates.append((ma, mb, arb_data["roi"]))
    
    print(f"  Found {len(candidates)} raw candidates")
    
    if not candidates:
        return []
    
    # Phase 3: Cross-Encoder reranking
    print("\n🎯 Phase 3: Nuance Verification (Cross-Encoder)...")
    candidates.sort(key=lambda x: x[2], reverse=True)
    candidates_to_rerank = candidates[:2000]
    
    pairs_to_score = [[c[0]['title'], c[1]['title']] for c in candidates_to_rerank]
    cross_scores = cross_model.predict(pairs_to_score, show_progress_bar=True, batch_size=64)
    
    # Build final results (with proper score handling from Apr 19 v2)
    final_pairs = []
    for i, score in enumerate(cross_scores):
        ma, mb, roi = candidates_to_rerank[i]
        # NLI model returns [contradiction, neutral, entailment] - get entailment (index 2)
        if hasattr(score, '__len__') and len(score) > 1:
            score_val = float(score[2])  # entailment score
        else:
            score_val = float(score)
        final_pairs.append({
            "marketA": ma, 
            "marketB": mb, 
            "roi": roi,
            "matchScore": min(100.0, max(0.0, score_val * 10.0)),
            "isVerified": bool(score_val > 5.0)
        })
    
    final_pairs.sort(key=lambda x: (x["isVerified"], x["roi"]), reverse=True)
    final_pairs = final_pairs[:top_k]
    
    elapsed = time.time() - start_time
    print(f"\n✅ ML Matching Complete! Verified {len(final_pairs)} matches in {elapsed:.2f}s")
    return final_pairs

# Execute matching
if all_markets:
    found_pairs = match_on_gpu(all_markets, min_similarity=55.0, min_roi=0.1, top_k=2000)
else:
    print("❌ No markets loaded")
    found_pairs = []

In [ ]:
# 5. Send results back via WebSocket
async def send_results_via_websocket(pairs):
    """Send matched pairs back to local backend via WebSocket"""
    print(f"\n📤 Sending {len(pairs)} matched pairs to local backend...")
    
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({
            "type": "cloud_results",
            "data": pairs,
            "count": len(pairs),
            "timestamp": time.time()
        }))
        
        # Wait for acknowledgment
        response = await ws.recv()
        ack = json.loads(response)
        
        if ack.get("type") == "results_received":
            print(f"✅ Backend confirmed receipt: {ack.get('message', '')}")
            return True
        else:
            print(f"⚠️ Unexpected acknowledgment: {ack}")
            return False

# Send results
if found_pairs:
    success = asyncio.get_event_loop().run_until_complete(
        send_results_via_websocket(found_pairs)
    )
    if success:
        print("\n🎉 Pipeline complete! Results sent to local backend for LLM verification.")
else:
    print("\n⚠️ No pairs to send")